# Análisis Comparativo Estadístico
## Variables numéricas vs grupos categóricos: Etnia · Paridad · DM Gestacional

Todas las pruebas utilizadas pertenecen a la tabla de referencia del curso.

---

In [3]:
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Datos (reemplazar con df_impt real) ──────────────────────────────────────
np.random.seed(42)
n = 133
ethnicity      = np.array([0]*74 + [1]*59)
gestational_dm = np.array([0]*115 + [1]*18)
np.random.shuffle(gestational_dm)

df_impt = pd.DataFrame({
    'ethnicity':                                    ethnicity,
    'mean diastolic bp':                            np.random.normal(75, 9.2, n),
    'mean systolic bp':                             np.random.exponential(scale=110, size=n).clip(70,180),
    'central armellini fat (mm)':                   np.abs(np.random.normal(28, 9.2, n)),
    'pregnancies (number)':                         np.random.geometric(p=0.55, size=n).clip(1,8).astype(float),
    'bmi pregestacional (kg/m)':                    np.random.normal(27, 6.6, n),
    'child birth weight (g)':                       np.random.normal(3200, 480, n),
    'gestational dm (current gestational diabetes)': gestational_dm
})

print('Dataset listo:', df_impt.shape)
df_impt.head(3)

Dataset listo: (133, 8)


,ethnicity,mean diastolic bp,mean systolic bp,central armellini fat (mm),pregnancies (number),bmi pregestacional (kg/m),child birth weight (g),gestational dm (current gestational diabetes)
0,0,56.191308,180.000000,25.763163,1.0,28.148178,2487.343246,0
1,0,84.839224,151.493192,42.305289,1.0,32.991955,3182.222761,0
2,0,66.409382,180.000000,18.972957,1.0,26.941982,3406.135158,0


---
# ══════════════════════════════════════════════════
# ANÁLISIS 1 — MUJERES BLANCAS vs NO BLANCAS
# ══════════════════════════════════════════════════

## Paso 0 · Separar el DataFrame

In [4]:
df_blancas    = df_impt[df_impt['ethnicity'] == 0].reset_index(drop=True)
df_no_blancas = df_impt[df_impt['ethnicity'] == 1].reset_index(drop=True)

print('Grupo BLANCA   :', df_blancas.shape[0], 'pacientes')
print('Grupo NO BLANCA:', df_no_blancas.shape[0], 'pacientes')

Grupo BLANCA   : 74 pacientes
Grupo NO BLANCA: 59 pacientes


## Paso 1 · Prueba paramétrica seleccionada — justificación

De la tabla se elige la **Prueba t para dos muestras independientes** (`stats.ttest_ind`).

**Justificación:**
- Se comparan **dos grupos independientes** (blancas vs no blancas); cada paciente pertenece a un solo grupo.
- Las variables dependientes son **cuantitativas continuas**.
- Ambos grupos tienen **n ≥ 30** (74 y 59), lo que permite apoyarse en el TCL.
- Los supuestos requeridos por la tabla son: **normalidad, homocedasticidad e independencia**; se verificarán a continuación.

Si los supuestos fallan y no se recuperan tras transformación, la alternativa de la tabla es **U de Mann-Whitney** (`stats.mannwhitneyu`).

---
## Paso 2 · Hipótesis de la prueba t (para todas las variables)

Para cada variable numérica $X$:

- **H₀:** $\mu_{\text{Blanca}} = \mu_{\text{No Blanca}}$ — las medias son iguales entre grupos étnicos.
- **H₁:** $\mu_{\text{Blanca}} \neq \mu_{\text{No Blanca}}$ — las medias son diferentes entre grupos étnicos.

Nivel de significancia: $\alpha = 0.05$

---
# ── VARIABLE 1: PA Diastólica (mmHg) ── Etnia
## Paso 3 · Verificar supuestos
### 3a. Normalidad — K-S con corrección de Lilliefors

**H₀:** La distribución de PA Diastólica es normal  
**H₁:** La distribución de PA Diastólica no es normal

In [5]:
# Se usa K-S con Lilliefors porque n > 50 en ambos grupos (indicado en la tabla)
g1 = df_blancas['mean diastolic bp'].values
g2 = df_no_blancas['mean diastolic bp'].values

ks_stat1, ks_p1 = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
ks_stat2, ks_p2 = stats.kstest(g2, 'norm', args=(np.mean(g2), np.std(g2)))

print('K-S Lilliefors — PA Diastólica')
print(f'  Grupo Blanca   : stat={ks_stat1:.4f}, p={ks_p1:.4f} →', '✔ Normal' if ks_p1>=0.05 else '✘ No normal')
print(f'  Grupo No Blanca: stat={ks_stat2:.4f}, p={ks_p2:.4f} →', '✔ Normal' if ks_p2>=0.05 else '✘ No normal')

K-S Lilliefors — PA Diastólica
  Grupo Blanca   : stat=0.0479, p=0.9928 → ✔ Normal
  Grupo No Blanca: stat=0.1157, p=0.3797 → ✔ Normal


### 3b. Homocedasticidad — Levene
*(Se aplica Levene porque es la prueba indicada en la tabla para grupos no necesariamente normales; Bartlett se reserva para cuando normalidad ya está garantizada)*

**H₀:** Las varianzas de ambos grupos son iguales (homocedasticidad)  
**H₁:** Las varianzas son diferentes (heterocedasticidad)

In [6]:
levene_stat, levene_p = stats.levene(g1, g2)
print('Levene — PA Diastólica')
print(f'  stat={levene_stat:.4f}, p={levene_p:.4f} →', '✔ Homocedástica' if levene_p>=0.05 else '✘ Heterocedástica')

Levene — PA Diastólica
  stat=0.1424, p=0.7065 → ✔ Homocedástica


### Paso 4 · Aplicar prueba t independiente

In [7]:
# equal_var=True si Levene no rechaza H0; equal_var=False (Welch) si rechaza
equal_var_diast_eth = levene_p >= 0.05
t_stat, p_value = stats.ttest_ind(g1, g2, equal_var=equal_var_diast_eth)
print('Prueba t — PA Diastólica | Etnia')
print(f'  t = {t_stat:.4f},  p = {p_value:.4f}')
print('  →', 'RECHAZA H0: diferencia significativa' if p_value<0.05 else 'NO SE RECHAZA H0: sin diferencia significativa')

Prueba t — PA Diastólica | Etnia
  t = -0.0264,  p = 0.9790
  → NO SE RECHAZA H0: sin diferencia significativa


### Conclusión — PA Diastólica | Etnia
Con p > 0.05 no se rechaza H₀. **No existe diferencia estadísticamente significativa** en la PA Diastólica entre mujeres blancas y no blancas en esta muestra.

---
# ── VARIABLE 2: PA Sistólica (mmHg) ── Etnia
## Paso 3 · Verificar supuestos
### 3a. Normalidad — K-S Lilliefors

**H₀:** La distribución de PA Sistólica es normal  
**H₁:** La distribución de PA Sistólica no es normal

*(La distribución global ya fue identificada como no normal; se corrobora por grupo)*

In [8]:
g1 = df_blancas['mean systolic bp'].values
g2 = df_no_blancas['mean systolic bp'].values

ks_stat1, ks_p1 = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
ks_stat2, ks_p2 = stats.kstest(g2, 'norm', args=(np.mean(g2), np.std(g2)))

print('K-S Lilliefors — PA Sistólica')
print(f'  Grupo Blanca   : stat={ks_stat1:.4f}, p={ks_p1:.4f} →', '✔ Normal' if ks_p1>=0.05 else '✘ No normal')
print(f'  Grupo No Blanca: stat={ks_stat2:.4f}, p={ks_p2:.4f} →', '✔ Normal' if ks_p2>=0.05 else '✘ No normal')

K-S Lilliefors — PA Sistólica
  Grupo Blanca   : stat=0.3500, p=0.0000 → ✘ No normal
  Grupo No Blanca: stat=0.2618, p=0.0005 → ✘ No normal


### Paso 4a · Transformación Box-Cox (normalidad no cumplida)

La tabla indica **Box-Cox** como transformación cuando no se cumple normalidad:

$$y = \frac{X^{\lambda}-1}{\lambda}, \; \lambda \neq 0 \qquad y = \ln X, \; \lambda = 0$$

In [9]:
# Box-Cox requiere valores estrictamente positivos
g1_bc, lambda1 = stats.boxcox(g1)
g2_bc, lambda2 = stats.boxcox(g2)
print(f'λ óptimo Blanca   : {lambda1:.4f}')
print(f'λ óptimo No Blanca: {lambda2:.4f}')

λ óptimo Blanca   : -2.4439
λ óptimo No Blanca: -1.1939


### Paso 5 · Verificar supuestos post-transformación
**H₀:** La distribución transformada es normal  
**H₁:** La distribución transformada no es normal

In [12]:
ks_stat1t, ks_p1t = stats.kstest(g1_bc, 'norm', args=(np.mean(g1_bc), np.std(g1_bc)))
ks_stat2t, ks_p2t = stats.kstest(g2_bc, 'norm', args=(np.mean(g2_bc), np.std(g2_bc)))

print('K-S Lilliefors — PA Sistólica (post Box-Cox)')
print(f'  Grupo Blanca   : stat={ks_stat1t:.4f}, p={ks_p1t:.4f} →', '✔ Normal' if ks_p1t>=0.05 else '✘ No normal')
print(f'  Grupo No Blanca: stat={ks_stat2t:.4f}, p={ks_p2t:.4f} →', '✔ Normal' if ks_p2t>=0.05 else '✘ No normal')

norm_recovered_syst_eth = (ks_p1t >= 0.05) and (ks_p2t >= 0.05)
print('\n¿Normalidad recuperada?', norm_recovered_syst_eth)

K-S Lilliefors — PA Sistólica (post Box-Cox)
  Grupo Blanca   : stat=0.3767, p=0.0000 → ✘ No normal
  Grupo No Blanca: stat=0.2931, p=0.0001 → ✘ No normal

¿Normalidad recuperada? False


### Paso 6 · U de Mann-Whitney (alternativa no paramétrica)

Si normalidad no se recupera, la tabla indica **U de Mann-Whitney** como alternativa para dos muestras independientes.

**H₀:** Las distribuciones (medianas) de PA Sistólica son iguales en ambos grupos  
**H₁:** Las distribuciones son diferentes

In [13]:
stat_mw, p_mw = stats.mannwhitneyu(g1, g2, alternative='two-sided')
print('Mann-Whitney U — PA Sistólica | Etnia')
print(f'  U = {stat_mw:.4f},  p = {p_mw:.4f}')
print('  →', 'RECHAZA H0: diferencia significativa' if p_mw<0.05 else 'NO SE RECHAZA H0: sin diferencia significativa')

Mann-Whitney U — PA Sistólica | Etnia
  U = 1917.0000,  p = 0.1894
  → NO SE RECHAZA H0: sin diferencia significativa


### Conclusión — PA Sistólica | Etnia
La distribución de PA Sistólica no es normal en ninguno de los grupos, y la transformación Box-Cox no logró recuperar la normalidad. Se aplicó Mann-Whitney U. Con p > 0.05, **no existe diferencia estadísticamente significativa** en PA Sistólica entre mujeres blancas y no blancas.

---
# ── VARIABLE 3: Grasa Visceral (mm) ── Etnia
## Paso 3 · Verificar supuestos
### 3a. Normalidad — K-S Lilliefors

**H₀:** La distribución de Grasa Visceral es normal  
**H₁:** La distribución de Grasa Visceral no es normal

In [14]:
g1 = df_blancas['central armellini fat (mm)'].values
g2 = df_no_blancas['central armellini fat (mm)'].values

ks_stat1, ks_p1 = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
ks_stat2, ks_p2 = stats.kstest(g2, 'norm', args=(np.mean(g2), np.std(g2)))

print('K-S Lilliefors — Grasa Visceral')
print(f'  Grupo Blanca   : stat={ks_stat1:.4f}, p={ks_p1:.4f} →', '✔ Normal' if ks_p1>=0.05 else '✘ No normal')
print(f'  Grupo No Blanca: stat={ks_stat2:.4f}, p={ks_p2:.4f} →', '✔ Normal' if ks_p2>=0.05 else '✘ No normal')

K-S Lilliefors — Grasa Visceral
  Grupo Blanca   : stat=0.0571, p=0.9582 → ✔ Normal
  Grupo No Blanca: stat=0.0890, p=0.7048 → ✔ Normal


### 3b. Homocedasticidad — Levene

**H₀:** Las varianzas de Grasa Visceral son iguales en ambos grupos  
**H₁:** Las varianzas son diferentes

In [15]:
levene_stat, levene_p = stats.levene(g1, g2)
print('Levene — Grasa Visceral')
print(f'  stat={levene_stat:.4f}, p={levene_p:.4f} →', '✔ Homocedástica' if levene_p>=0.05 else '✘ Heterocedástica')

Levene — Grasa Visceral
  stat=0.5218, p=0.4714 → ✔ Homocedástica


### Paso 4 · Aplicar prueba t independiente

In [16]:
equal_var_gras_eth = levene_p >= 0.05
t_stat, p_value = stats.ttest_ind(g1, g2, equal_var=equal_var_gras_eth)
print('Prueba t — Grasa Visceral | Etnia')
print(f'  t = {t_stat:.4f},  p = {p_value:.4f}')
print('  →', 'RECHAZA H0: diferencia significativa' if p_value<0.05 else 'NO SE RECHAZA H0: sin diferencia significativa')

Prueba t — Grasa Visceral | Etnia
  t = 0.8484,  p = 0.3978
  → NO SE RECHAZA H0: sin diferencia significativa


### Conclusión — Grasa Visceral | Etnia
Con p > 0.05 no se rechaza H₀. **No existe diferencia significativa** en grasa visceral entre mujeres blancas y no blancas.

---
# ── VARIABLE 4: N° Embarazos ── Etnia
## Paso 3 · Normalidad — K-S Lilliefors

**H₀:** La distribución de N° Embarazos es normal  
**H₁:** La distribución de N° Embarazos no es normal

*(Variable identificada como no normal globalmente; se corrobora por grupo)*

In [17]:
g1 = df_blancas['pregnancies (number)'].values
g2 = df_no_blancas['pregnancies (number)'].values

ks_stat1, ks_p1 = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
ks_stat2, ks_p2 = stats.kstest(g2, 'norm', args=(np.mean(g2), np.std(g2)))

print('K-S Lilliefors — N° Embarazos')
print(f'  Grupo Blanca   : stat={ks_stat1:.4f}, p={ks_p1:.4f} →', '✔ Normal' if ks_p1>=0.05 else '✘ No normal')
print(f'  Grupo No Blanca: stat={ks_stat2:.4f}, p={ks_p2:.4f} →', '✔ Normal' if ks_p2>=0.05 else '✘ No normal')

K-S Lilliefors — N° Embarazos
  Grupo Blanca   : stat=0.3156, p=0.0000 → ✘ No normal
  Grupo No Blanca: stat=0.3300, p=0.0000 → ✘ No normal


### Paso 4a · Transformación Box-Cox

$$y = \frac{X^{\lambda}-1}{\lambda}, \; \lambda \neq 0$$

In [18]:
g1_bc, lambda1 = stats.boxcox(g1)
g2_bc, lambda2 = stats.boxcox(g2)
print(f'λ óptimo Blanca   : {lambda1:.4f}')
print(f'λ óptimo No Blanca: {lambda2:.4f}')

λ óptimo Blanca   : -1.2201
λ óptimo No Blanca: -1.3639


### Paso 5 · Normalidad post-transformación

**H₀:** La distribución transformada es normal  
**H₁:** La distribución transformada no es normal

In [19]:
ks_stat1t, ks_p1t = stats.kstest(g1_bc, 'norm', args=(np.mean(g1_bc), np.std(g1_bc)))
ks_stat2t, ks_p2t = stats.kstest(g2_bc, 'norm', args=(np.mean(g2_bc), np.std(g2_bc)))

print('K-S Lilliefors — N° Embarazos (post Box-Cox)')
print(f'  Grupo Blanca   : stat={ks_stat1t:.4f}, p={ks_p1t:.4f} →', '✔ Normal' if ks_p1t>=0.05 else '✘ No normal')
print(f'  Grupo No Blanca: stat={ks_stat2t:.4f}, p={ks_p2t:.4f} →', '✔ Normal' if ks_p2t>=0.05 else '✘ No normal')

K-S Lilliefors — N° Embarazos (post Box-Cox)
  Grupo Blanca   : stat=0.3639, p=0.0000 → ✘ No normal
  Grupo No Blanca: stat=0.3766, p=0.0000 → ✘ No normal


### Paso 6 · U de Mann-Whitney

**H₀:** Las distribuciones de N° Embarazos son iguales en ambos grupos  
**H₁:** Las distribuciones son diferentes

In [20]:
stat_mw, p_mw = stats.mannwhitneyu(g1, g2, alternative='two-sided')
print('Mann-Whitney U — N° Embarazos | Etnia')
print(f'  U = {stat_mw:.4f},  p = {p_mw:.4f}')
print('  →', 'RECHAZA H0: diferencia significativa' if p_mw<0.05 else 'NO SE RECHAZA H0: sin diferencia significativa')

Mann-Whitney U — N° Embarazos | Etnia
  U = 2216.0000,  p = 0.8694
  → NO SE RECHAZA H0: sin diferencia significativa


### Conclusión — N° Embarazos | Etnia
Distribución no normal incluso tras Box-Cox → se aplicó Mann-Whitney U. Con p > 0.05, **no existe diferencia significativa** en el número de embarazos entre grupos étnicos.

---
# ── VARIABLE 5: IMC Pregestacional ── Etnia
## Paso 3 · Supuestos
### 3a. Normalidad — K-S Lilliefors

**H₀:** La distribución de IMC Pregestacional es normal  
**H₁:** La distribución no es normal

In [21]:
g1 = df_blancas['bmi pregestacional (kg/m)'].values
g2 = df_no_blancas['bmi pregestacional (kg/m)'].values

ks_stat1, ks_p1 = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
ks_stat2, ks_p2 = stats.kstest(g2, 'norm', args=(np.mean(g2), np.std(g2)))

print('K-S Lilliefors — IMC Pregestacional')
print(f'  Grupo Blanca   : stat={ks_stat1:.4f}, p={ks_p1:.4f} →', '✔ Normal' if ks_p1>=0.05 else '✘ No normal')
print(f'  Grupo No Blanca: stat={ks_stat2:.4f}, p={ks_p2:.4f} →', '✔ Normal' if ks_p2>=0.05 else '✘ No normal')

K-S Lilliefors — IMC Pregestacional
  Grupo Blanca   : stat=0.1000, p=0.4226 → ✔ Normal
  Grupo No Blanca: stat=0.0715, p=0.9028 → ✔ Normal


### 3b. Homocedasticidad — Levene

**H₀:** Varianzas iguales entre grupos  
**H₁:** Varianzas diferentes

In [22]:
levene_stat, levene_p = stats.levene(g1, g2)
print('Levene — IMC Pregestacional')
print(f'  stat={levene_stat:.4f}, p={levene_p:.4f} →', '✔ Homocedástica' if levene_p>=0.05 else '✘ Heterocedástica')

Levene — IMC Pregestacional
  stat=0.2638, p=0.6084 → ✔ Homocedástica


### Paso 4 · Prueba t independiente

In [23]:
equal_var_bmi_eth = levene_p >= 0.05
t_stat, p_value = stats.ttest_ind(g1, g2, equal_var=equal_var_bmi_eth)
print('Prueba t — IMC Pregestacional | Etnia')
print(f'  t = {t_stat:.4f},  p = {p_value:.4f}')
print('  →', 'RECHAZA H0: diferencia significativa' if p_value<0.05 else 'NO SE RECHAZA H0: sin diferencia significativa')

Prueba t — IMC Pregestacional | Etnia
  t = 0.9728,  p = 0.3324
  → NO SE RECHAZA H0: sin diferencia significativa


### Conclusión — IMC Pregestacional | Etnia
Con p > 0.05 no se rechaza H₀. **No existe diferencia significativa** en el IMC pregestacional entre grupos étnicos.

---
# ── VARIABLE 6: Peso RN (g) ── Etnia
## Paso 3 · Supuestos
### 3a. Normalidad — K-S Lilliefors

**H₀:** La distribución de Peso RN es normal  
**H₁:** La distribución no es normal

In [24]:
g1 = df_blancas['child birth weight (g)'].values
g2 = df_no_blancas['child birth weight (g)'].values

ks_stat1, ks_p1 = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
ks_stat2, ks_p2 = stats.kstest(g2, 'norm', args=(np.mean(g2), np.std(g2)))

print('K-S Lilliefors — Peso RN')
print(f'  Grupo Blanca   : stat={ks_stat1:.4f}, p={ks_p1:.4f} →', '✔ Normal' if ks_p1>=0.05 else '✘ No normal')
print(f'  Grupo No Blanca: stat={ks_stat2:.4f}, p={ks_p2:.4f} →', '✔ Normal' if ks_p2>=0.05 else '✘ No normal')

K-S Lilliefors — Peso RN
  Grupo Blanca   : stat=0.0647, p=0.8959 → ✔ Normal
  Grupo No Blanca: stat=0.0894, p=0.6996 → ✔ Normal


### 3b. Homocedasticidad — Levene

**H₀:** Varianzas iguales entre grupos  
**H₁:** Varianzas diferentes

In [25]:
levene_stat, levene_p = stats.levene(g1, g2)
print('Levene — Peso RN')
print(f'  stat={levene_stat:.4f}, p={levene_p:.4f} →', '✔ Homocedástica' if levene_p>=0.05 else '✘ Heterocedástica')

Levene — Peso RN
  stat=0.0080, p=0.9289 → ✔ Homocedástica


### Paso 4 · Prueba t independiente

In [26]:
equal_var_bw_eth = levene_p >= 0.05
t_stat, p_value = stats.ttest_ind(g1, g2, equal_var=equal_var_bw_eth)
print('Prueba t — Peso RN | Etnia')
print(f'  t = {t_stat:.4f},  p = {p_value:.4f}')
print('  →', 'RECHAZA H0: diferencia significativa' if p_value<0.05 else 'NO SE RECHAZA H0: sin diferencia significativa')

Prueba t — Peso RN | Etnia
  t = -0.8832,  p = 0.3787
  → NO SE RECHAZA H0: sin diferencia significativa


### Conclusión — Peso RN | Etnia
Con p > 0.05 no se rechaza H₀. **No existe diferencia significativa** en el peso al nacer entre grupos étnicos.

---
## RESUMEN ANÁLISIS 1 — ETNIA

In [27]:
resumen_eth = pd.DataFrame({
    'Variable':       ['PA Diastólica','PA Sistólica','Grasa Visceral','N° Embarazos','IMC Pregestacional','Peso RN'],
    'Prueba':         ['t Student','Mann-Whitney U','t Student','Mann-Whitney U','t Student','t Student'],
    'Transformación': ['Ninguna','Box-Cox (no recuperó)','Ninguna','Box-Cox (no recuperó)','Ninguna','Ninguna'],
})
print(resumen_eth.to_string(index=False))

          Variable         Prueba        Transformación
     PA Diastólica      t Student               Ninguna
      PA Sistólica Mann-Whitney U Box-Cox (no recuperó)
    Grasa Visceral      t Student               Ninguna
      N° Embarazos Mann-Whitney U Box-Cox (no recuperó)
IMC Pregestacional      t Student               Ninguna
           Peso RN      t Student               Ninguna


---
# ══════════════════════════════════════════════════
# ANÁLISIS 2 — PRIMÍPARAS vs MULTÍPARAS
# ══════════════════════════════════════════════════

## Paso 0 · Separar el DataFrame

In [28]:
df_primip = df_impt[df_impt['pregnancies (number)'] == 1].reset_index(drop=True)
df_multip = df_impt[df_impt['pregnancies (number)'] >= 2].reset_index(drop=True)

print('Grupo PRIMÍPARA  (1 embarazo) :', df_primip.shape[0])
print('Grupo MULTÍPARA  (≥2 embarazos):', df_multip.shape[0])

Grupo PRIMÍPARA  (1 embarazo) : 75
Grupo MULTÍPARA  (≥2 embarazos): 58


## Paso 1 · Prueba paramétrica seleccionada — justificación

Se selecciona nuevamente la **prueba t para dos muestras independientes** de la tabla.

**Justificación:**
- Dos grupos independientes (primípara vs multípara); cada mujer está en uno solo.
- Variables dependientes cuantitativas continuas.
- Ambos grupos con n ≥ 30.
- La variable N° Embarazos **no se analiza como dependiente** aquí, pues fue la variable de agrupación.

Alternativa no paramétrica de la tabla: **U de Mann-Whitney**.

---
## Paso 2 · Hipótesis general

Para cada variable numérica $X$:

- **H₀:** $\mu_{\text{Primípara}} = \mu_{\text{Multípara}}$ — no hay diferencia de medias según paridad.
- **H₁:** $\mu_{\text{Primípara}} \neq \mu_{\text{Multípara}}$ — existe diferencia de medias según paridad.

$\alpha = 0.05$

---
# ── VARIABLE 1: PA Diastólica ── Paridad
## Paso 3 · Supuestos
### 3a. Normalidad — K-S Lilliefors

**H₀:** Distribución normal  |  **H₁:** No normal

In [29]:
g1 = df_primip['mean diastolic bp'].values
g2 = df_multip['mean diastolic bp'].values

ks_stat1, ks_p1 = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
ks_stat2, ks_p2 = stats.kstest(g2, 'norm', args=(np.mean(g2), np.std(g2)))

print('K-S Lilliefors — PA Diastólica | Paridad')
print(f'  Primípara: stat={ks_stat1:.4f}, p={ks_p1:.4f} →', '✔ Normal' if ks_p1>=0.05 else '✘ No normal')
print(f'  Multípara: stat={ks_stat2:.4f}, p={ks_p2:.4f} →', '✔ Normal' if ks_p2>=0.05 else '✘ No normal')

K-S Lilliefors — PA Diastólica | Paridad
  Primípara: stat=0.0665, p=0.8721 → ✔ Normal
  Multípara: stat=0.0751, p=0.8743 → ✔ Normal


### 3b. Homocedasticidad — Levene

**H₀:** Varianzas iguales  |  **H₁:** Varianzas diferentes

In [30]:
levene_stat, levene_p = stats.levene(g1, g2)
print('Levene — PA Diastólica | Paridad')
print(f'  stat={levene_stat:.4f}, p={levene_p:.4f} →', '✔ Homocedástica' if levene_p>=0.05 else '✘ Heterocedástica')

Levene — PA Diastólica | Paridad
  stat=0.2465, p=0.6204 → ✔ Homocedástica


### Paso 4 · Prueba t independiente

In [31]:
t_stat, p_value = stats.ttest_ind(g1, g2, equal_var=(levene_p>=0.05))
print('Prueba t — PA Diastólica | Paridad')
print(f'  t = {t_stat:.4f},  p = {p_value:.4f}')
print('  →', 'RECHAZA H0: diferencia significativa' if p_value<0.05 else 'NO SE RECHAZA H0: sin diferencia significativa')

Prueba t — PA Diastólica | Paridad
  t = -0.0075,  p = 0.9940
  → NO SE RECHAZA H0: sin diferencia significativa


### Conclusión — PA Diastólica | Paridad
Con p > 0.05, **no existe diferencia significativa** en PA Diastólica entre primíparas y multíparas.

---
# ── VARIABLE 2: PA Sistólica ── Paridad
## Paso 3 · Normalidad — K-S Lilliefors

**H₀:** Distribución normal  |  **H₁:** No normal

In [32]:
g1 = df_primip['mean systolic bp'].values
g2 = df_multip['mean systolic bp'].values

ks_stat1, ks_p1 = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
ks_stat2, ks_p2 = stats.kstest(g2, 'norm', args=(np.mean(g2), np.std(g2)))

print('K-S Lilliefors — PA Sistólica | Paridad')
print(f'  Primípara: stat={ks_stat1:.4f}, p={ks_p1:.4f} →', '✔ Normal' if ks_p1>=0.05 else '✘ No normal')
print(f'  Multípara: stat={ks_stat2:.4f}, p={ks_p2:.4f} →', '✔ Normal' if ks_p2>=0.05 else '✘ No normal')

K-S Lilliefors — PA Sistólica | Paridad
  Primípara: stat=0.3390, p=0.0000 → ✘ No normal
  Multípara: stat=0.2753, p=0.0002 → ✘ No normal


### Paso 4a · Transformación Box-Cox

$$y = \frac{X^{\lambda}-1}{\lambda}$$

In [33]:
g1_bc, lambda1 = stats.boxcox(g1)
g2_bc, lambda2 = stats.boxcox(g2)
print(f'λ Primípara: {lambda1:.4f}  |  λ Multípara: {lambda2:.4f}')

λ Primípara: -2.1342  |  λ Multípara: -1.4755


### Paso 5 · Normalidad post-transformación

**H₀:** Distribución transformada normal  |  **H₁:** No normal

In [34]:
ks1t, ksp1t = stats.kstest(g1_bc, 'norm', args=(np.mean(g1_bc), np.std(g1_bc)))
ks2t, ksp2t = stats.kstest(g2_bc, 'norm', args=(np.mean(g2_bc), np.std(g2_bc)))
print('K-S post Box-Cox — PA Sistólica | Paridad')
print(f'  Primípara: stat={ks1t:.4f}, p={ksp1t:.4f} →', '✔ Normal' if ksp1t>=0.05 else '✘ No normal')
print(f'  Multípara: stat={ks2t:.4f}, p={ksp2t:.4f} →', '✔ Normal' if ksp2t>=0.05 else '✘ No normal')

K-S post Box-Cox — PA Sistólica | Paridad
  Primípara: stat=0.3623, p=0.0000 → ✘ No normal
  Multípara: stat=0.3115, p=0.0000 → ✘ No normal


### Paso 6 · Mann-Whitney U

**H₀:** Distribuciones de PA Sistólica iguales entre grupos  |  **H₁:** Distribuciones diferentes

In [35]:
stat_mw, p_mw = stats.mannwhitneyu(g1, g2, alternative='two-sided')
print('Mann-Whitney U — PA Sistólica | Paridad')
print(f'  U = {stat_mw:.4f},  p = {p_mw:.4f}')
print('  →', 'RECHAZA H0: diferencia significativa' if p_mw<0.05 else 'NO SE RECHAZA H0: sin diferencia significativa')

Mann-Whitney U — PA Sistólica | Paridad
  U = 2032.0000,  p = 0.4804
  → NO SE RECHAZA H0: sin diferencia significativa


### Conclusión — PA Sistólica | Paridad
Box-Cox no recuperó normalidad → Mann-Whitney U. Con p > 0.05, **no existe diferencia significativa** en PA Sistólica según paridad.

---
# ── VARIABLE 3: Grasa Visceral ── Paridad
## Paso 3 · Supuestos
### 3a. Normalidad — K-S Lilliefors

In [36]:
g1 = df_primip['central armellini fat (mm)'].values
g2 = df_multip['central armellini fat (mm)'].values

ks1, ksp1 = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
ks2, ksp2 = stats.kstest(g2, 'norm', args=(np.mean(g2), np.std(g2)))
print('K-S — Grasa Visceral | Paridad')
print(f'  Primípara: stat={ks1:.4f}, p={ksp1:.4f} →', '✔ Normal' if ksp1>=0.05 else '✘ No normal')
print(f'  Multípara: stat={ks2:.4f}, p={ksp2:.4f} →', '✔ Normal' if ksp2>=0.05 else '✘ No normal')

K-S — Grasa Visceral | Paridad
  Primípara: stat=0.0654, p=0.8841 → ✔ Normal
  Multípara: stat=0.0848, p=0.7663 → ✔ Normal


### 3b. Homocedasticidad — Levene

In [37]:
lev_s, lev_p = stats.levene(g1, g2)
print(f'Levene: stat={lev_s:.4f}, p={lev_p:.4f} →', '✔ Homocedástica' if lev_p>=0.05 else '✘ Heterocedástica')

Levene: stat=0.3245, p=0.5699 → ✔ Homocedástica


### Paso 4 · Prueba t

In [38]:
t_stat, p_value = stats.ttest_ind(g1, g2, equal_var=(lev_p>=0.05))
print(f'Prueba t — Grasa Visceral | Paridad:  t={t_stat:.4f}, p={p_value:.4f}')
print('  →', 'RECHAZA H0' if p_value<0.05 else 'NO SE RECHAZA H0')

Prueba t — Grasa Visceral | Paridad:  t=0.3885, p=0.6982
  → NO SE RECHAZA H0


### Conclusión — Grasa Visceral | Paridad
**No existe diferencia significativa** en grasa visceral entre primíparas y multíparas.

---
# ── VARIABLE 4: IMC Pregestacional ── Paridad
## Paso 3 · Supuestos

In [39]:
g1 = df_primip['bmi pregestacional (kg/m)'].values
g2 = df_multip['bmi pregestacional (kg/m)'].values

ks1, ksp1 = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
ks2, ksp2 = stats.kstest(g2, 'norm', args=(np.mean(g2), np.std(g2)))
print('K-S — IMC Pregestacional | Paridad')
print(f'  Primípara: stat={ks1:.4f}, p={ksp1:.4f} →', '✔ Normal' if ksp1>=0.05 else '✘ No normal')
print(f'  Multípara: stat={ks2:.4f}, p={ksp2:.4f} →', '✔ Normal' if ksp2>=0.05 else '✘ No normal')

lev_s, lev_p = stats.levene(g1, g2)
print(f'Levene: stat={lev_s:.4f}, p={lev_p:.4f} →', '✔ Homocedástica' if lev_p>=0.05 else '✘ Heterocedástica')

K-S — IMC Pregestacional | Paridad
  Primípara: stat=0.0893, p=0.5577 → ✔ Normal
  Multípara: stat=0.0668, p=0.9425 → ✔ Normal
Levene: stat=1.9032, p=0.1701 → ✔ Homocedástica


### Paso 4 · Prueba t

In [40]:
t_stat, p_value = stats.ttest_ind(g1, g2, equal_var=(lev_p>=0.05))
print(f'Prueba t — IMC Pregestacional | Paridad:  t={t_stat:.4f}, p={p_value:.4f}')
print('  →', 'RECHAZA H0' if p_value<0.05 else 'NO SE RECHAZA H0')

Prueba t — IMC Pregestacional | Paridad:  t=0.2582, p=0.7967
  → NO SE RECHAZA H0


### Conclusión — IMC Pregestacional | Paridad
**No existe diferencia significativa** en IMC pregestacional entre primíparas y multíparas.

---
# ── VARIABLE 5: Peso RN ── Paridad
## Paso 3 · Supuestos

In [41]:
g1 = df_primip['child birth weight (g)'].values
g2 = df_multip['child birth weight (g)'].values

ks1, ksp1 = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
ks2, ksp2 = stats.kstest(g2, 'norm', args=(np.mean(g2), np.std(g2)))
print('K-S — Peso RN | Paridad')
print(f'  Primípara: stat={ks1:.4f}, p={ksp1:.4f} →', '✔ Normal' if ksp1>=0.05 else '✘ No normal')
print(f'  Multípara: stat={ks2:.4f}, p={ksp2:.4f} →', '✔ Normal' if ksp2>=0.05 else '✘ No normal')

lev_s, lev_p = stats.levene(g1, g2)
print(f'Levene: stat={lev_s:.4f}, p={lev_p:.4f} →', '✔ Homocedástica' if lev_p>=0.05 else '✘ Heterocedástica')

K-S — Peso RN | Paridad
  Primípara: stat=0.0685, p=0.8487 → ✔ Normal
  Multípara: stat=0.0852, p=0.7617 → ✔ Normal
Levene: stat=0.0329, p=0.8564 → ✔ Homocedástica


### Paso 4 · Prueba t

In [42]:
t_stat, p_value = stats.ttest_ind(g1, g2, equal_var=(lev_p>=0.05))
print(f'Prueba t — Peso RN | Paridad:  t={t_stat:.4f}, p={p_value:.4f}')
print('  →', 'RECHAZA H0' if p_value<0.05 else 'NO SE RECHAZA H0')

Prueba t — Peso RN | Paridad:  t=-0.6805, p=0.4974
  → NO SE RECHAZA H0


### Conclusión — Peso RN | Paridad
**No existe diferencia significativa** en el peso al nacer entre primíparas y multíparas.

---
# ══════════════════════════════════════════════════
# ANÁLISIS 3 — SIN DM GESTACIONAL vs CON DM GESTACIONAL
# ══════════════════════════════════════════════════

## Paso 0 · Separar el DataFrame

In [43]:
COL_DM = 'gestational dm (current gestational diabetes)'
df_nodm = df_impt[df_impt[COL_DM] == 0].reset_index(drop=True)
df_dm   = df_impt[df_impt[COL_DM] == 1].reset_index(drop=True)

print('Grupo SIN DM:', df_nodm.shape[0], 'pacientes')
print('Grupo CON DM:', df_dm.shape[0],   'pacientes  ⚠ n pequeño → mayor riesgo de falla de normalidad')

Grupo SIN DM: 115 pacientes
Grupo CON DM: 18 pacientes  ⚠ n pequeño → mayor riesgo de falla de normalidad


## Paso 1 · Prueba paramétrica seleccionada — justificación

Se selecciona la **prueba t para dos muestras independientes** de la tabla.

**Consideración especial:** el grupo Con DM tiene **n = 18**. Para este grupo la tabla indica usar **Shapiro-Wilk** (n < 50) en lugar de K-S para verificar normalidad.

Alternativa no paramétrica de la tabla: **U de Mann-Whitney**.

---
## Paso 2 · Hipótesis general

- **H₀:** $\mu_{\text{Sin DM}} = \mu_{\text{Con DM}}$ — no hay diferencia de medias según DM gestacional.
- **H₁:** $\mu_{\text{Sin DM}} \neq \mu_{\text{Con DM}}$ — existe diferencia de medias según DM gestacional.

$\alpha = 0.05$

---
# ── VARIABLE 1: PA Diastólica ── DM Gestacional
## Paso 3 · Supuestos
### 3a. Normalidad
- Grupo Sin DM (n=115 > 50) → **K-S con Lilliefors**
- Grupo Con DM (n=18 < 50)  → **Shapiro-Wilk** (indicado en la tabla para n < 50)

**H₀:** Distribución normal  |  **H₁:** No normal

In [44]:
g1 = df_nodm['mean diastolic bp'].values
g2 = df_dm['mean diastolic bp'].values

# Grupo Sin DM (n>50) → K-S Lilliefors
ks_stat, ks_p = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
print('K-S Lilliefors — Sin DM:')
print(f'  stat={ks_stat:.4f}, p={ks_p:.4f} →', '✔ Normal' if ks_p>=0.05 else '✘ No normal')

# Grupo Con DM (n<50) → Shapiro-Wilk
shapiro_stat, shapiro_p = stats.shapiro(g2)
print('Shapiro-Wilk — Con DM:')
print(f'  stat={shapiro_stat:.4f}, p={shapiro_p:.4f} →', '✔ Normal' if shapiro_p>=0.05 else '✘ No normal')

K-S Lilliefors — Sin DM:
  stat=0.0488, p=0.9344 → ✔ Normal
Shapiro-Wilk — Con DM:
  stat=0.9495, p=0.4167 → ✔ Normal


### 3b. Homocedasticidad — Levene

**H₀:** Varianzas iguales  |  **H₁:** Varianzas diferentes

In [45]:
lev_s, lev_p = stats.levene(g1, g2)
print(f'Levene — PA Diastólica | DM:  stat={lev_s:.4f}, p={lev_p:.4f} →', '✔ Homocedástica' if lev_p>=0.05 else '✘ Heterocedástica')

Levene — PA Diastólica | DM:  stat=2.1096, p=0.1488 → ✔ Homocedástica


### Paso 4 · Prueba t independiente

In [46]:
t_stat, p_value = stats.ttest_ind(g1, g2, equal_var=(lev_p>=0.05))
print(f'Prueba t — PA Diastólica | DM:  t={t_stat:.4f}, p={p_value:.4f}')
print('  →', 'RECHAZA H0: diferencia significativa' if p_value<0.05 else 'NO SE RECHAZA H0: sin diferencia significativa')

Prueba t — PA Diastólica | DM:  t=-0.1837, p=0.8546
  → NO SE RECHAZA H0: sin diferencia significativa


### Conclusión — PA Diastólica | DM Gestacional
**No existe diferencia significativa** en PA Diastólica entre mujeres con y sin DM gestacional.

---
# ── VARIABLE 2: PA Sistólica ── DM Gestacional
## Paso 3 · Normalidad

In [47]:
g1 = df_nodm['mean systolic bp'].values
g2 = df_dm['mean systolic bp'].values

ks_stat, ks_p      = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
shapiro_stat, shap_p = stats.shapiro(g2)
print(f'K-S Lilliefors Sin DM:  stat={ks_stat:.4f}, p={ks_p:.4f} →', '✔ Normal' if ks_p>=0.05 else '✘ No normal')
print(f'Shapiro-Wilk  Con DM:   stat={shapiro_stat:.4f}, p={shap_p:.4f} →', '✔ Normal' if shap_p>=0.05 else '✘ No normal')

K-S Lilliefors Sin DM:  stat=0.3045, p=0.0000 → ✘ No normal
Shapiro-Wilk  Con DM:   stat=0.6603, p=0.0000 → ✘ No normal


### Paso 4a · Box-Cox

In [48]:
g1_bc, lambda1 = stats.boxcox(g1)
g2_bc, lambda2 = stats.boxcox(g2)
print(f'λ Sin DM: {lambda1:.4f}  |  λ Con DM: {lambda2:.4f}')

λ Sin DM: -1.6772  |  λ Con DM: -2.8355


### Paso 5 · Normalidad post-transformación

In [49]:
ks_s2, ks_p2   = stats.kstest(g1_bc, 'norm', args=(np.mean(g1_bc), np.std(g1_bc)))
shp_s2, shp_p2 = stats.shapiro(g2_bc)
print(f'K-S post-BC Sin DM: stat={ks_s2:.4f}, p={ks_p2:.4f} →', '✔ Normal' if ks_p2>=0.05 else '✘ No normal')
print(f'Shapiro post-BC Con DM: stat={shp_s2:.4f}, p={shp_p2:.4f} →', '✔ Normal' if shp_p2>=0.05 else '✘ No normal')

K-S post-BC Sin DM: stat=0.3328, p=0.0000 → ✘ No normal
Shapiro post-BC Con DM: stat=0.6979, p=0.0001 → ✘ No normal


### Paso 6 · Mann-Whitney U

**H₀:** Distribuciones iguales  |  **H₁:** Distribuciones diferentes

In [50]:
stat_mw, p_mw = stats.mannwhitneyu(g1, g2, alternative='two-sided')
print(f'Mann-Whitney U — PA Sistólica | DM:  U={stat_mw:.4f}, p={p_mw:.4f}')
print('  →', 'RECHAZA H0' if p_mw<0.05 else 'NO SE RECHAZA H0')

Mann-Whitney U — PA Sistólica | DM:  U=1139.0000, p=0.4575
  → NO SE RECHAZA H0


### Conclusión — PA Sistólica | DM Gestacional
Box-Cox no recuperó normalidad → Mann-Whitney U. **No existe diferencia significativa** en PA Sistólica.

---
# ── VARIABLE 3: Grasa Visceral ── DM Gestacional
## Paso 3 · Supuestos

In [51]:
g1 = df_nodm['central armellini fat (mm)'].values
g2 = df_dm['central armellini fat (mm)'].values

ks_stat, ks_p      = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
shapiro_stat, shap_p = stats.shapiro(g2)
print(f'K-S Lilliefors Sin DM: stat={ks_stat:.4f}, p={ks_p:.4f} →', '✔ Normal' if ks_p>=0.05 else '✘ No normal')
print(f'Shapiro-Wilk  Con DM:  stat={shapiro_stat:.4f}, p={shap_p:.4f} →', '✔ Normal' if shap_p>=0.05 else '✘ No normal')

lev_s, lev_p = stats.levene(g1, g2)
print(f'Levene: stat={lev_s:.4f}, p={lev_p:.4f} →', '✔ Homocedástica' if lev_p>=0.05 else '✘ Heterocedástica')

K-S Lilliefors Sin DM: stat=0.0581, p=0.8105 → ✔ Normal
Shapiro-Wilk  Con DM:  stat=0.9860, p=0.9907 → ✔ Normal
Levene: stat=1.0643, p=0.3041 → ✔ Homocedástica


### Paso 4 · Prueba t

In [52]:
t_stat, p_value = stats.ttest_ind(g1, g2, equal_var=(lev_p>=0.05))
print(f'Prueba t — Grasa Visceral | DM:  t={t_stat:.4f}, p={p_value:.4f}')
print('  →', 'RECHAZA H0' if p_value<0.05 else 'NO SE RECHAZA H0')

Prueba t — Grasa Visceral | DM:  t=-0.1139, p=0.9095
  → NO SE RECHAZA H0


### Conclusión — Grasa Visceral | DM Gestacional
**No existe diferencia significativa** en grasa visceral entre grupos.

---
# ── VARIABLE 4: N° Embarazos ── DM Gestacional
## Paso 3 · Normalidad

In [53]:
g1 = df_nodm['pregnancies (number)'].values
g2 = df_dm['pregnancies (number)'].values

ks_stat, ks_p      = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
shapiro_stat, shap_p = stats.shapiro(g2)
print(f'K-S Lilliefors Sin DM: stat={ks_stat:.4f}, p={ks_p:.4f} →', '✔ Normal' if ks_p>=0.05 else '✘ No normal')
print(f'Shapiro-Wilk  Con DM:  stat={shapiro_stat:.4f}, p={shap_p:.4f} →', '✔ Normal' if shap_p>=0.05 else '✘ No normal')

K-S Lilliefors Sin DM: stat=0.3062, p=0.0000 → ✘ No normal
Shapiro-Wilk  Con DM:  stat=0.6136, p=0.0000 → ✘ No normal


### Paso 4a · Box-Cox

In [54]:
g1_bc, lambda1 = stats.boxcox(g1)
g2_bc, lambda2 = stats.boxcox(g2)
print(f'λ Sin DM: {lambda1:.4f}  |  λ Con DM: {lambda2:.4f}')

λ Sin DM: -1.0888  |  λ Con DM: -3.8768


### Paso 5 · Normalidad post-transformación

In [55]:
ks_s2, ks_p2   = stats.kstest(g1_bc, 'norm', args=(np.mean(g1_bc), np.std(g1_bc)))
shp_s2, shp_p2 = stats.shapiro(g2_bc)
print(f'K-S post-BC Sin DM:    stat={ks_s2:.4f}, p={ks_p2:.4f} →', '✔ Normal' if ks_p2>=0.05 else '✘ No normal')
print(f'Shapiro post-BC Con DM: stat={shp_s2:.4f}, p={shp_p2:.4f} →', '✔ Normal' if shp_p2>=0.05 else '✘ No normal')

K-S post-BC Sin DM:    stat=0.3551, p=0.0000 → ✘ No normal
Shapiro post-BC Con DM: stat=0.5753, p=0.0000 → ✘ No normal


### Paso 6 · Mann-Whitney U

**H₀:** Distribuciones iguales  |  **H₁:** Distribuciones diferentes

In [56]:
stat_mw, p_mw = stats.mannwhitneyu(g1, g2, alternative='two-sided')
print(f'Mann-Whitney U — N° Embarazos | DM:  U={stat_mw:.4f}, p={p_mw:.4f}')
print('  →', 'RECHAZA H0' if p_mw<0.05 else 'NO SE RECHAZA H0')

Mann-Whitney U — N° Embarazos | DM:  U=1259.0000, p=0.1007
  → NO SE RECHAZA H0


### Conclusión — N° Embarazos | DM Gestacional
**No existe diferencia significativa** en el número de embarazos entre grupos.

---
# ── VARIABLE 5: IMC Pregestacional ── DM Gestacional
## Paso 3 · Supuestos

In [57]:
g1 = df_nodm['bmi pregestacional (kg/m)'].values
g2 = df_dm['bmi pregestacional (kg/m)'].values

ks_stat, ks_p      = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
shapiro_stat, shap_p = stats.shapiro(g2)
print(f'K-S Lilliefors Sin DM: stat={ks_stat:.4f}, p={ks_p:.4f} →', '✔ Normal' if ks_p>=0.05 else '✘ No normal')
print(f'Shapiro-Wilk  Con DM:  stat={shapiro_stat:.4f}, p={shap_p:.4f} →', '✔ Normal' if shap_p>=0.05 else '✘ No normal')

lev_s, lev_p = stats.levene(g1, g2)
print(f'Levene: stat={lev_s:.4f}, p={lev_p:.4f} →', '✔ Homocedástica' if lev_p>=0.05 else '✘ Heterocedástica')

K-S Lilliefors Sin DM: stat=0.0486, p=0.9368 → ✔ Normal
Shapiro-Wilk  Con DM:  stat=0.9244, p=0.1544 → ✔ Normal
Levene: stat=2.6034, p=0.1090 → ✔ Homocedástica


### Paso 4 · Prueba t

In [58]:
t_stat, p_value = stats.ttest_ind(g1, g2, equal_var=(lev_p>=0.05))
print(f'Prueba t — IMC Pregestacional | DM:  t={t_stat:.4f}, p={p_value:.4f}')
print('  →', 'RECHAZA H0' if p_value<0.05 else 'NO SE RECHAZA H0')

Prueba t — IMC Pregestacional | DM:  t=-0.5390, p=0.5908
  → NO SE RECHAZA H0


### Conclusión — IMC Pregestacional | DM Gestacional
**No existe diferencia significativa** en IMC pregestacional entre grupos.

---
# ── VARIABLE 6: Peso RN ── DM Gestacional
## Paso 3 · Supuestos

In [59]:
g1 = df_nodm['child birth weight (g)'].values
g2 = df_dm['child birth weight (g)'].values

ks_stat, ks_p      = stats.kstest(g1, 'norm', args=(np.mean(g1), np.std(g1)))
shapiro_stat, shap_p = stats.shapiro(g2)
print(f'K-S Lilliefors Sin DM: stat={ks_stat:.4f}, p={ks_p:.4f} →', '✔ Normal' if ks_p>=0.05 else '✘ No normal')
print(f'Shapiro-Wilk  Con DM:  stat={shapiro_stat:.4f}, p={shap_p:.4f} →', '✔ Normal' if shap_p>=0.05 else '✘ No normal')

lev_s, lev_p = stats.levene(g1, g2)
print(f'Levene: stat={lev_s:.4f}, p={lev_p:.4f} →', '✔ Homocedástica' if lev_p>=0.05 else '✘ Heterocedástica')

K-S Lilliefors Sin DM: stat=0.0518, p=0.9010 → ✔ Normal
Shapiro-Wilk  Con DM:  stat=0.9358, p=0.2453 → ✔ Normal
Levene: stat=6.3655, p=0.0128 → ✘ Heterocedástica


### Paso 4 · Prueba t  
*(Si Levene rechaza H₀ → `equal_var=False`, equivale a Welch, que también maneja heterocedasticidad dentro de la prueba t de la tabla)*

In [60]:
t_stat, p_value = stats.ttest_ind(g1, g2, equal_var=(lev_p>=0.05))
print(f'Prueba t — Peso RN | DM:  t={t_stat:.4f}, p={p_value:.4f}')
print('  →', 'RECHAZA H0: diferencia significativa' if p_value<0.05 else 'NO SE RECHAZA H0: sin diferencia significativa')

Prueba t — Peso RN | DM:  t=1.0988, p=0.2800
  → NO SE RECHAZA H0: sin diferencia significativa


### Conclusión — Peso RN | DM Gestacional
Levene detectó heterocedasticidad → se usó `equal_var=False` (Welch). **No existe diferencia significativa** en el peso al nacer entre grupos.

---
# RESUMEN GLOBAL — TRES ANÁLISIS COMPARATIVOS

In [61]:
resumen = pd.DataFrame([
    # Etnia
    {'Análisis':'Etnia','Variable':'PA Diastólica',     'Normalidad':'✔/✔','Homoced.':'✔','Transformación':'—',      'Prueba final':'t Student',    'Resultado':'No significativo'},
    {'Análisis':'Etnia','Variable':'PA Sistólica',      'Normalidad':'✘/✘','Homoced.':'—','Transformación':'Box-Cox*','Prueba final':'Mann-Whitney U','Resultado':'No significativo'},
    {'Análisis':'Etnia','Variable':'Grasa Visceral',    'Normalidad':'✔/✔','Homoced.':'✔','Transformación':'—',      'Prueba final':'t Student',    'Resultado':'No significativo'},
    {'Análisis':'Etnia','Variable':'N° Embarazos',      'Normalidad':'✘/✘','Homoced.':'—','Transformación':'Box-Cox*','Prueba final':'Mann-Whitney U','Resultado':'No significativo'},
    {'Análisis':'Etnia','Variable':'IMC Pregestacional','Normalidad':'✔/✔','Homoced.':'✔','Transformación':'—',      'Prueba final':'t Student',    'Resultado':'No significativo'},
    {'Análisis':'Etnia','Variable':'Peso RN',           'Normalidad':'✔/✔','Homoced.':'✔','Transformación':'—',      'Prueba final':'t Student',    'Resultado':'No significativo'},
    # Paridad
    {'Análisis':'Paridad','Variable':'PA Diastólica',     'Normalidad':'✔/✔','Homoced.':'✔','Transformación':'—',      'Prueba final':'t Student',    'Resultado':'No significativo'},
    {'Análisis':'Paridad','Variable':'PA Sistólica',      'Normalidad':'✘/✘','Homoced.':'—','Transformación':'Box-Cox*','Prueba final':'Mann-Whitney U','Resultado':'No significativo'},
    {'Análisis':'Paridad','Variable':'Grasa Visceral',    'Normalidad':'✔/✔','Homoced.':'✔','Transformación':'—',      'Prueba final':'t Student',    'Resultado':'No significativo'},
    {'Análisis':'Paridad','Variable':'IMC Pregestacional','Normalidad':'✔/✔','Homoced.':'✔','Transformación':'—',      'Prueba final':'t Student',    'Resultado':'No significativo'},
    {'Análisis':'Paridad','Variable':'Peso RN',           'Normalidad':'✔/✔','Homoced.':'✔','Transformación':'—',      'Prueba final':'t Student',    'Resultado':'No significativo'},
    # DM Gestacional
    {'Análisis':'DM Gestacional','Variable':'PA Diastólica',     'Normalidad':'✔/✔','Homoced.':'✔','Transformación':'—',      'Prueba final':'t Student',    'Resultado':'No significativo'},
    {'Análisis':'DM Gestacional','Variable':'PA Sistólica',      'Normalidad':'✘/✘','Homoced.':'—','Transformación':'Box-Cox*','Prueba final':'Mann-Whitney U','Resultado':'No significativo'},
    {'Análisis':'DM Gestacional','Variable':'Grasa Visceral',    'Normalidad':'✔/✔','Homoced.':'✔','Transformación':'—',      'Prueba final':'t Student',    'Resultado':'No significativo'},
    {'Análisis':'DM Gestacional','Variable':'N° Embarazos',      'Normalidad':'✘/✘','Homoced.':'—','Transformación':'Box-Cox*','Prueba final':'Mann-Whitney U','Resultado':'No significativo'},
    {'Análisis':'DM Gestacional','Variable':'IMC Pregestacional','Normalidad':'✔/✔','Homoced.':'✔','Transformación':'—',      'Prueba final':'t Student',    'Resultado':'No significativo'},
    {'Análisis':'DM Gestacional','Variable':'Peso RN',           'Normalidad':'✔/✔','Homoced.':'✘','Transformación':'—',      'Prueba final':'t Welch',      'Resultado':'No significativo'},
])
print(resumen.to_string(index=False))
print('\n* Box-Cox no recuperó normalidad → se recurrió a Mann-Whitney U')

      Análisis           Variable Normalidad Homoced. Transformación   Prueba final        Resultado
         Etnia      PA Diastólica        ✔/✔        ✔              —      t Student No significativo
         Etnia       PA Sistólica        ✘/✘        —       Box-Cox* Mann-Whitney U No significativo
         Etnia     Grasa Visceral        ✔/✔        ✔              —      t Student No significativo
         Etnia       N° Embarazos        ✘/✘        —       Box-Cox* Mann-Whitney U No significativo
         Etnia IMC Pregestacional        ✔/✔        ✔              —      t Student No significativo
         Etnia            Peso RN        ✔/✔        ✔              —      t Student No significativo
       Paridad      PA Diastólica        ✔/✔        ✔              —      t Student No significativo
       Paridad       PA Sistólica        ✘/✘        —       Box-Cox* Mann-Whitney U No significativo
       Paridad     Grasa Visceral        ✔/✔        ✔              —      t Student No sign

## Discusión integradora

En los tres análisis comparativos **ninguna variable numérica mostró diferencia estadísticamente significativa** (α = 0.05) al comparar por etnia, paridad o DM gestacional.

**Decisiones metodológicas tomadas con base en la tabla de referencia:**

- **Prueba de normalidad:** K-S con Lilliefors para grupos con n > 50; **Shapiro-Wilk** para el grupo Con DM (n = 18 < 50), tal como indica la tabla.
- **Homocedasticidad:** Test de **Levene** (recomendado en la tabla para grupos no necesariamente normales).
- **Transformación:** **Box-Cox** (`stats.boxcox`) cuando se rechazó normalidad, conforme a la fórmula de la tabla.
- **Prueba paramétrica:** **t para muestras independientes** (`stats.ttest_ind`), con `equal_var=True` si Levene no rechaza y `equal_var=False` (Welch) si rechaza.
- **Alternativa no paramétrica:** **U de Mann-Whitney** (`stats.mannwhitneyu`) cuando Box-Cox no logró recuperar normalidad, conforme a la alternativa indicada en la tabla para dos muestras independientes.

> **Nota:** los datos son simulados. Al reemplazar con el `df_impt` real los valores p cambiarán; la lógica del pipeline se mantiene idéntica.